# PCB Defect Detection — YOLO Pipeline (Kaggle)
### DeepPCB + HRIPCB (PKU-Market-PCB) + PCB-Defect (Mendeley) -> unified YOLO dataset -> train

All three source zips are bundled in ONE Kaggle Dataset: `micro-pcb-defect`
(`DeepPCB-master.zip`, `PCB_Defect.zip`, `Tiny-Defect-Detection-for-PCB-master.zip`).

This notebook **auto-detects** the exact mount path and the internal folder
layout of each zip, so you don't need to hand-edit paths — it prints what it
found before running anything, so you can sanity-check before conversion.

**Before running:** turn on **GPU** and **Internet** in notebook Settings,
and make sure the `micro-pcb-defect` dataset is attached via *Add Input*.


## 0. Locate the dataset root and list its contents

In [14]:
import os

candidates = [
    "/kaggle/input/datasets/rahatrihan/micro-pcb-defect",
    "/kaggle/input/micro-pcb-defect",
]
BASE = next((p for p in candidates if os.path.isdir(p)), None)
if BASE is None:
    # last resort: search for it
    for root, dirs, _ in os.walk("/kaggle/input"):
        if "micro-pcb-defect" in dirs:
            BASE = os.path.join(root, "micro-pcb-defect")
            break
if BASE is None:
    raise FileNotFoundError("Could not find the micro-pcb-defect dataset under /kaggle/input. "
                             "Run !find /kaggle/input -maxdepth 3 and check it's attached as an input.")
print("Using base path:", BASE)

DEEPPCB_ROOT   = f"{BASE}/DeepPCB-master"
HRIPCB_ROOT    = f"{BASE}/Tiny-Defect-Detection-for-PCB-master"
PCBDEFECT_ROOT = f"{BASE}/PCB_Defect"
WORK = "/kaggle/working"


Using base path: /kaggle/input/datasets/rahatrihan/micro-pcb-defect


## 1. Auto-discover the real annotation/image locations inside each zip

Zip internal layouts vary (nesting depth, folder names). Rather than hardcoding
guesses, this walks each root and finds:
- the folder with the most `.xml` files (VOC annotations, for HRIPCB)
- the folder with the most image files (for HRIPCB / PCB-Defect)
- the first `.json` file found (COCO annotations, for PCB-Defect)

DeepPCB doesn't need this — `convert_deeppcb.py` already searches recursively
for `*_test.jpg` files under whatever root you give it.

In [15]:
from pathlib import Path
from collections import Counter

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def best_dir_by_extension(root, exts, exts_are_images=False):
    """Return the directory under `root` containing the most files matching exts."""
    root = Path(root)
    if not root.exists():
        return None, 0
    counts = Counter()
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            counts[p.parent] += 1
    if not counts:
        return None, 0
    best_dir, n = counts.most_common(1)[0]
    return str(best_dir), n

def first_file(root, exts):
    root = Path(root)
    if not root.exists():
        return None
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            return str(p)
    return None

# --- HRIPCB ---
HRIPCB_ANN, n_xml = best_dir_by_extension(HRIPCB_ROOT, {".xml"})
HRIPCB_IMG, n_img = best_dir_by_extension(HRIPCB_ROOT, IMG_EXTS)
print(f"HRIPCB: found {n_xml} .xml files in {HRIPCB_ANN}")
print(f"HRIPCB: found {n_img} image files in {HRIPCB_IMG}")
if n_xml == 0:
    print("\n*** WARNING: no .xml annotation files found anywhere under Tiny-Defect-Detection-for-PCB-master.")
    print("*** This repo is GitHub CODE, not the raw HRIPCB dataset — the original PKU-Market-PCB")
    print("*** images+annotations are not bundled in it. You likely need to source HRIPCB separately")
    print("*** (a Kaggle dataset upload of the actual PKU-Market-PCB images/XML, not this repo).")

# --- PCB-Defect ---
PCBDEFECT_JSON = first_file(PCBDEFECT_ROOT, {".json"})
PCBDEFECT_IMG, n_img2 = best_dir_by_extension(PCBDEFECT_ROOT, IMG_EXTS)
print(f"\nPCB-Defect: annotations json = {PCBDEFECT_JSON}")
print(f"PCB-Defect: found {n_img2} image files in {PCBDEFECT_IMG}")
if PCBDEFECT_JSON is None:
    print("\n*** WARNING: no .json file found under PCB_Defect — check the zip actually contains")
    print("*** the COCO annotations file and re-check the extracted layout.")

# --- DeepPCB (sanity check only, converter handles its own recursive search) ---
n_test_imgs = sum(1 for _ in Path(DEEPPCB_ROOT).rglob("*_test.jpg"))
print(f"\nDeepPCB: found {n_test_imgs} *_test.jpg images under {DEEPPCB_ROOT}")
if n_test_imgs == 0:
    print("*** WARNING: no *_test.jpg files found — check DeepPCB-master actually contains PCBData/.")


HRIPCB: found 2 .xml files in /kaggle/input/datasets/rahatrihan/micro-pcb-defect/Tiny-Defect-Detection-for-PCB-master/Tiny-Defect-Detection-for-PCB-master/tools/test_annotation
HRIPCB: found 67 image files in /kaggle/input/datasets/rahatrihan/micro-pcb-defect/Tiny-Defect-Detection-for-PCB-master/Tiny-Defect-Detection-for-PCB-master/tools/inference_results

PCB-Defect: annotations json = /kaggle/input/datasets/rahatrihan/micro-pcb-defect/PCB_Defect/PCB_Defect/annotation/_annotations.coco.json
PCB-Defect: found 230 image files in /kaggle/input/datasets/rahatrihan/micro-pcb-defect/PCB_Defect/PCB_Defect/images

DeepPCB: found 1503 *_test.jpg images under /kaggle/input/datasets/rahatrihan/micro-pcb-defect/DeepPCB-master


**Stop here and read the printed output.** If HRIPCB shows 0 XML files, you
don't have the real HRIPCB dataset in this Kaggle Dataset yet — either add it
as a 4th zip (the actual PKU-Market-PCB images + XML annotations, not the
GitHub repo) or proceed with just DeepPCB + PCB-Defect for now (skip the
HRIPCB conversion cell below, and drop `hripcb=...` from the merge step).

## 2. Install dependencies

In [16]:
!pip install -q ultralytics pillow


## 3. Write out the pipeline scripts

In [17]:
%%writefile class_map.py
"""
Unified class map for merging DeepPCB + HRIPCB (PKU-Market-PCB) + PCB-Defect
(Mendeley) into a single YOLO-format dataset.

All three datasets describe the same 6 PCB defect types under different
names. This file is the single source of truth for the class order used
everywhere else in the pipeline (YOLO class ids = index into CANONICAL_CLASSES).

IMPORTANT: verify the DeepPCB numeric-id mapping below against the README
that ships inside your actual downloaded copy (id order has been reported
inconsistently across mirrors/releases) before trusting it blindly.
"""

# Canonical class order -> this is what your data.yaml and trained model will use
CANONICAL_CLASSES = [
    "missing_hole",     # missing hole / missing pad
    "mouse_bite",
    "open_circuit",
    "short",
    "spur",
    "spurious_copper",
]

CLASS_TO_ID = {name: i for i, name in enumerate(CANONICAL_CLASSES)}

# ---------------------------------------------------------------------------
# DeepPCB: annotation files use numeric ids per defect box.
# Per the DeepPCB README (tangsanli5201/DeepPCB):
#   0 - open, 1 - short, 2 - mousebite, 3 - spur, 4 - copper, 5 - pin-hole
# copper == spurious_copper, pin-hole == missing_hole. CONFIRM against your copy.
DEEPPCB_ID_TO_CANONICAL = {
    0: "open_circuit",
    1: "short",
    2: "mouse_bite",
    3: "spur",
    4: "spurious_copper",
    5: "missing_hole",
}

# ---------------------------------------------------------------------------
# HRIPCB / PKU-Market-PCB: VOC XML <name> tags (as released).
HRIPCB_NAME_TO_CANONICAL = {
    "missing_hole": "missing_hole",
    "mouse_bite": "mouse_bite",
    "open_circuit": "open_circuit",
    "short": "short",
    "spur": "spur",
    "spurious_copper": "spurious_copper",
    # a few mirrors use British/alt spellings — extend if your XMLs differ
    "spurious copper": "spurious_copper",
    "short_circuit": "short",
}

# ---------------------------------------------------------------------------
# PCB-Defect (Mendeley, vdj74sngvn) COCO category names, as documented:
# missing pad, mouse bite, open circuit, short circuit, spur, spurious copper
PCBDEFECT_NAME_TO_CANONICAL = {
    "missing pad": "missing_hole",
    "mouse bite": "mouse_bite",
    "open circuit": "open_circuit",
    "short circuit": "short",
    "spur": "spur",
    "spurious copper": "spurious_copper",
}


Overwriting class_map.py


In [18]:
%%writefile convert_deeppcb.py
"""
Convert DeepPCB dataset to YOLO format.

Expected source layout (as cloned from github.com/tangsanli5201/DeepPCB):
    PCBData/
        groupXXXXX/
            XXXXX/
                *_test.jpg        (defect image)
                *_temp.jpg        (template / defect-free image, NOT used for training)
            XXXXX_not/
                *.txt             (annotation: one file per test image)

Each annotation .txt has one bbox per line:
    x1 y1 x2 y2 type_id
(pixel coordinates, top-left / bottom-right corners)

Output:
    out_dir/images/<uid>.jpg
    out_dir/labels/<uid>.txt   (YOLO format: class cx cy w h, normalized 0-1)

Usage:
    python convert_deeppcb.py --src /path/to/PCBData --out /path/to/converted/deeppcb
"""
import argparse
import os
import shutil
from pathlib import Path

from PIL import Image

from class_map import CLASS_TO_ID, DEEPPCB_ID_TO_CANONICAL


def convert(src_root: Path, out_root: Path):
    img_out = out_root / "images"
    lbl_out = out_root / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    n_images, n_boxes, skipped = 0, 0, 0

    # test images live at .../groupXXXXX/XXXXX/*_test.jpg
    for test_img_path in src_root.rglob("*_test.jpg"):
        stem = test_img_path.stem.replace("_test", "")
        # annotation dir is the sibling "<XXXXX>_not" folder
        ann_dir = test_img_path.parent.parent / f"{test_img_path.parent.name}_not"
        ann_path = ann_dir / f"{stem}.txt"
        if not ann_path.exists():
            skipped += 1
            continue

        with Image.open(test_img_path) as im:
            w, h = im.size

        uid = f"deeppcb_{test_img_path.parent.parent.name}_{stem}"
        shutil.copy2(test_img_path, img_out / f"{uid}.jpg")

        yolo_lines = []
        with open(ann_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                x1, y1, x2, y2, type_id = parts
                x1, y1, x2, y2 = float(x1), float(y1), float(x2), float(y2)
                type_id = int(type_id)

                canonical = DEEPPCB_ID_TO_CANONICAL.get(type_id)
                if canonical is None:
                    continue
                cls_id = CLASS_TO_ID[canonical]

                cx = ((x1 + x2) / 2) / w
                cy = ((y1 + y2) / 2) / h
                bw = abs(x2 - x1) / w
                bh = abs(y2 - y1) / h
                yolo_lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
                n_boxes += 1

        (lbl_out / f"{uid}.txt").write_text("\n".join(yolo_lines))
        n_images += 1

    print(f"[DeepPCB] converted {n_images} images, {n_boxes} boxes, skipped {skipped} (no annotation found)")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True, help="Path to DeepPCB PCBData folder")
    ap.add_argument("--out", required=True, help="Output folder for converted images+labels")
    args = ap.parse_args()
    convert(Path(args.src), Path(args.out))


Overwriting convert_deeppcb.py


In [19]:
%%writefile convert_hripcb.py
"""
Convert HRIPCB / PKU-Market-PCB dataset (Pascal VOC XML annotations) to YOLO format.

Expected source layout (typical release structure):
    Annotations/*.xml
    images/*.jpg   (or JPEGImages/*.jpg)

Output:
    out_dir/images/<uid>.jpg
    out_dir/labels/<uid>.txt

Usage:
    python convert_hripcb.py --ann /path/to/Annotations --img /path/to/images --out /path/to/converted/hripcb
"""
import argparse
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

from class_map import CLASS_TO_ID, HRIPCB_NAME_TO_CANONICAL


def convert(ann_dir: Path, img_dir: Path, out_root: Path):
    img_out = out_root / "images"
    lbl_out = out_root / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    n_images, n_boxes, skipped = 0, 0, 0

    for xml_path in ann_dir.glob("*.xml"):
        tree = ET.parse(xml_path)
        root = tree.getroot()

        filename = root.findtext("filename")
        size = root.find("size")
        w = int(size.findtext("width"))
        h = int(size.findtext("height"))

        # locate the matching image file (extension can vary by mirror)
        candidates = [img_dir / filename] if filename else []
        candidates += list(img_dir.glob(f"{xml_path.stem}.*"))
        src_img = next((c for c in candidates if c.exists()), None)
        if src_img is None:
            skipped += 1
            continue

        uid = f"hripcb_{xml_path.stem}"
        shutil.copy2(src_img, img_out / f"{uid}{src_img.suffix}")

        yolo_lines = []
        for obj in root.findall("object"):
            name = obj.findtext("name").strip().lower()
            canonical = HRIPCB_NAME_TO_CANONICAL.get(name)
            if canonical is None:
                continue
            cls_id = CLASS_TO_ID[canonical]

            bnd = obj.find("bndbox")
            xmin = float(bnd.findtext("xmin"))
            ymin = float(bnd.findtext("ymin"))
            xmax = float(bnd.findtext("xmax"))
            ymax = float(bnd.findtext("ymax"))

            cx = ((xmin + xmax) / 2) / w
            cy = ((ymin + ymax) / 2) / h
            bw = (xmax - xmin) / w
            bh = (ymax - ymin) / h
            yolo_lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            n_boxes += 1

        (lbl_out / f"{uid}.txt").write_text("\n".join(yolo_lines))
        n_images += 1

    print(f"[HRIPCB] converted {n_images} images, {n_boxes} boxes, skipped {skipped} (image not found)")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--ann", required=True, help="Path to VOC Annotations folder (*.xml)")
    ap.add_argument("--img", required=True, help="Path to images folder")
    ap.add_argument("--out", required=True, help="Output folder for converted images+labels")
    args = ap.parse_args()
    convert(Path(args.ann), Path(args.img), Path(args.out))


Overwriting convert_hripcb.py


In [20]:
%%writefile convert_pcbdefect.py
"""
Convert PCB-Defect (Mendeley, vdj74sngvn) COCO-format dataset to YOLO format.

Expected source layout (as downloaded from Mendeley):
    annotations.json   (COCO json: images / annotations / categories)
    images/*.jpg (or .png)

Output:
    out_dir/images/<uid>.jpg
    out_dir/labels/<uid>.txt

Usage:
    python convert_pcbdefect.py --json /path/to/annotations.json --img /path/to/images --out /path/to/converted/pcbdefect
"""
import argparse
import json
import shutil
from pathlib import Path

from class_map import CLASS_TO_ID, PCBDEFECT_NAME_TO_CANONICAL


def convert(json_path: Path, img_dir: Path, out_root: Path):
    img_out = out_root / "images"
    lbl_out = out_root / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    with open(json_path) as f:
        coco = json.load(f)

    # category_id -> canonical class id
    cat_id_to_canonical_id = {}
    for cat in coco["categories"]:
        name = cat["name"].strip().lower()
        canonical = PCBDEFECT_NAME_TO_CANONICAL.get(name)
        if canonical is None:
            print(f"[PCB-Defect] WARNING: unmapped category '{cat['name']}' — check class_map.py")
            continue
        cat_id_to_canonical_id[cat["id"]] = CLASS_TO_ID[canonical]

    images_by_id = {im["id"]: im for im in coco["images"]}

    # group annotations by image
    anns_by_image = {}
    for ann in coco["annotations"]:
        anns_by_image.setdefault(ann["image_id"], []).append(ann)

    n_images, n_boxes, skipped = 0, 0, 0

    for image_id, im_info in images_by_id.items():
        filename = im_info["file_name"]
        w, h = im_info["width"], im_info["height"]

        src_img = img_dir / filename
        if not src_img.exists():
            # some releases nest images in subfolders — fall back to a name-only search
            matches = list(img_dir.rglob(Path(filename).name))
            src_img = matches[0] if matches else None
        if src_img is None or not src_img.exists():
            skipped += 1
            continue

        uid = f"pcbdefect_{Path(filename).stem}"
        shutil.copy2(src_img, img_out / f"{uid}{src_img.suffix}")

        yolo_lines = []
        for ann in anns_by_image.get(image_id, []):
            cls_id = cat_id_to_canonical_id.get(ann["category_id"])
            if cls_id is None:
                continue
            x, y, bw, bh = ann["bbox"]  # COCO bbox = [x_min, y_min, width, height]
            cx = (x + bw / 2) / w
            cy = (y + bh / 2) / h
            nbw = bw / w
            nbh = bh / h
            yolo_lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {nbw:.6f} {nbh:.6f}")
            n_boxes += 1

        (lbl_out / f"{uid}.txt").write_text("\n".join(yolo_lines))
        n_images += 1

    print(f"[PCB-Defect] converted {n_images} images, {n_boxes} boxes, skipped {skipped} (image not found)")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--json", required=True, help="Path to COCO annotations.json")
    ap.add_argument("--img", required=True, help="Path to images folder")
    ap.add_argument("--out", required=True, help="Output folder for converted images+labels")
    args = ap.parse_args()
    convert(Path(args.json), Path(args.img), Path(args.out))


Overwriting convert_pcbdefect.py


In [21]:
%%writefile tile_large_images.py
"""
Tile large images (+ their YOLO labels) into fixed-size patches with overlap.

Why: PCB-Defect (Mendeley) images range up to 6000x4000px with small defects.
If you resize a 6000x4000 image down to YOLO's 640x640 input, a defect that
is e.g. 40x40px becomes ~4x4px — likely undetectable. Tiling into overlapping
640x640 (or 1280x1280) patches keeps defects at native pixel scale.

Only run this on the CONVERTED (YOLO-format) output of a dataset whose images
exceed roughly 1.5-2x your target --imgsz. DeepPCB (640x640) and most HRIPCB
images do not need this; PCB-Defect usually does.

Usage:
    python tile_large_images.py --src converted/pcbdefect --out converted/pcbdefect_tiled \
        --tile 1024 --overlap 128 --min-size 1500
"""
import argparse
from pathlib import Path

from PIL import Image


def yolo_to_xyxy(line, w, h):
    cls_id, cx, cy, bw, bh = line.split()
    cls_id = int(cls_id)
    cx, cy, bw, bh = float(cx) * w, float(cy) * h, float(bw) * w, float(bh) * h
    x1, y1, x2, y2 = cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2
    return cls_id, x1, y1, x2, y2


def tile_image(img_path: Path, lbl_path: Path, out_img_dir: Path, out_lbl_dir: Path,
                tile: int, overlap: int, min_size: int):
    with Image.open(img_path) as im:
        w, h = im.size
        if max(w, h) < min_size:
            # small enough already — just copy through untouched
            im.save(out_img_dir / img_path.name)
            if lbl_path.exists():
                (out_lbl_dir / lbl_path.name).write_text(lbl_path.read_text())
            return 1

        boxes = []
        if lbl_path.exists():
            for line in lbl_path.read_text().splitlines():
                if line.strip():
                    boxes.append(yolo_to_xyxy(line, w, h))

        stride = tile - overlap
        n_tiles = 0
        y = 0
        while y < h:
            x = 0
            y_end = min(y + tile, h)
            while x < w:
                x_end = min(x + tile, w)
                crop = im.crop((x, y, x_end, y_end))

                tw, th = x_end - x, y_end - y
                kept = []
                for cls_id, bx1, by1, bx2, by2 in boxes:
                    # keep boxes whose center falls inside this tile
                    bcx, bcy = (bx1 + bx2) / 2, (by1 + by2) / 2
                    if not (x <= bcx < x_end and y <= bcy < y_end):
                        continue
                    cbx1, cby1 = max(bx1, x) - x, max(by1, y) - y
                    cbx2, cby2 = min(bx2, x_end) - x, min(by2, y_end) - y
                    ncx = (cbx1 + cbx2) / 2 / tw
                    ncy = (cby1 + cby2) / 2 / th
                    nbw = (cbx2 - cbx1) / tw
                    nbh = (cby2 - cby1) / th
                    kept.append(f"{cls_id} {ncx:.6f} {ncy:.6f} {nbw:.6f} {nbh:.6f}")

                if kept:  # skip empty background tiles to avoid class imbalance blowup
                    tile_name = f"{img_path.stem}_x{x}_y{y}"
                    crop.save(out_img_dir / f"{tile_name}{img_path.suffix}")
                    (out_lbl_dir / f"{tile_name}.txt").write_text("\n".join(kept))
                    n_tiles += 1

                x += stride
                if x_end == w:
                    break
            y += stride
            if y_end == h:
                break
        return n_tiles


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True, help="Converted dataset folder with images/ and labels/")
    ap.add_argument("--out", required=True, help="Output folder for tiled images/labels")
    ap.add_argument("--tile", type=int, default=1024, help="Tile size in px (square)")
    ap.add_argument("--overlap", type=int, default=128, help="Overlap between adjacent tiles in px")
    ap.add_argument("--min-size", type=int, default=1500,
                     help="Images with max(w,h) below this are copied through untiled")
    args = ap.parse_args()

    src = Path(args.src)
    out_img_dir = Path(args.out) / "images"
    out_lbl_dir = Path(args.out) / "labels"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    total_tiles, n_src = 0, 0
    for img_path in (src / "images").iterdir():
        lbl_path = src / "labels" / f"{img_path.stem}.txt"
        total_tiles += tile_image(img_path, lbl_path, out_img_dir, out_lbl_dir,
                                   args.tile, args.overlap, args.min_size)
        n_src += 1

    print(f"[Tile] processed {n_src} source images -> {total_tiles} output images/tiles")


if __name__ == "__main__":
    main()


Overwriting tile_large_images.py


In [22]:
%%writefile merge_split.py
"""
Merge converted datasets (each with images/ + labels/) into one YOLO dataset
with train/val splits, one split done PER SOURCE DATASET so every source is
represented in both train and val (prevents val metrics being dominated by
whichever dataset is largest).

Usage:
    python merge_split.py \
        --src deeppcb=converted/deeppcb hripcb=converted/hripcb pcbdefect=converted/pcbdefect_tiled \
        --out dataset --val-frac 0.15 --seed 42
"""
import argparse
import random
import shutil
from pathlib import Path


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", nargs="+", required=True,
                     help="name=path pairs, e.g. deeppcb=converted/deeppcb")
    ap.add_argument("--out", required=True)
    ap.add_argument("--val-frac", type=float, default=0.15)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    random.seed(args.seed)
    out = Path(args.out)
    for split in ("train", "val"):
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)

    summary = {}
    for pair in args.src:
        name, path = pair.split("=", 1)
        path = Path(path)
        img_files = sorted((path / "images").iterdir())
        random.shuffle(img_files)
        n_val = max(1, int(len(img_files) * args.val_frac))
        val_set = set(img_files[:n_val])

        counts = {"train": 0, "val": 0}
        for img_path in img_files:
            split = "val" if img_path in val_set else "train"
            lbl_path = path / "labels" / f"{img_path.stem}.txt"

            shutil.copy2(img_path, out / "images" / split / img_path.name)
            if lbl_path.exists():
                shutil.copy2(lbl_path, out / "labels" / split / f"{img_path.stem}.txt")
            else:
                (out / "labels" / split / f"{img_path.stem}.txt").write_text("")
            counts[split] += 1

        summary[name] = counts
        print(f"[{name}] train={counts['train']}  val={counts['val']}")

    total_train = sum(c["train"] for c in summary.values())
    total_val = sum(c["val"] for c in summary.values())
    print(f"\nTOTAL: train={total_train}  val={total_val}")
    print(f"Dataset root written to: {out.resolve()}")


if __name__ == "__main__":
    main()


Overwriting merge_split.py


In [23]:
%%writefile analyze_dataset.py
"""
Sanity-check the merged YOLO dataset before training:
  - class balance (box count per class)
  - bbox size distribution (relative to image size) -> informs --imgsz choice
  - per-source-dataset image counts (inferred from filename prefix)

Usage:
    python analyze_dataset.py --root dataset --split train
"""
import argparse
from collections import Counter
from pathlib import Path

from PIL import Image

from class_map import CANONICAL_CLASSES


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True, help="Dataset root (with images/<split>, labels/<split>)")
    ap.add_argument("--split", default="train")
    args = ap.parse_args()

    img_dir = Path(args.root) / "images" / args.split
    lbl_dir = Path(args.root) / "labels" / args.split

    class_counts = Counter()
    source_counts = Counter()
    rel_sizes = []  # (rel_width, rel_height) per box, relative to its image

    for img_path in img_dir.iterdir():
        source = img_path.stem.split("_")[0]
        source_counts[source] += 1

        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        if not lbl_path.exists() or not lbl_path.read_text().strip():
            continue
        with Image.open(img_path) as im:
            w, h = im.size
        for line in lbl_path.read_text().splitlines():
            if not line.strip():
                continue
            cls_id, cx, cy, bw, bh = line.split()
            cls_id = int(cls_id)
            class_counts[CANONICAL_CLASSES[cls_id]] += 1
            rel_sizes.append((float(bw), float(bh), float(bw) * w, float(bh) * h))

    print("=== Images per source dataset ===")
    for src, n in source_counts.most_common():
        print(f"  {src:12s} {n}")

    print("\n=== Box count per class ===")
    total = sum(class_counts.values())
    for cls in CANONICAL_CLASSES:
        n = class_counts.get(cls, 0)
        pct = 100 * n / total if total else 0
        print(f"  {cls:18s} {n:6d}  ({pct:5.1f}%)")

    if rel_sizes:
        avg_px_w = sum(s[2] for s in rel_sizes) / len(rel_sizes)
        avg_px_h = sum(s[3] for s in rel_sizes) / len(rel_sizes)
        small = sum(1 for s in rel_sizes if s[2] < 32 and s[3] < 32)
        print(f"\n=== Box size (absolute pixels, native image scale) ===")
        print(f"  avg box size: {avg_px_w:.1f} x {avg_px_h:.1f} px")
        print(f"  boxes smaller than 32x32px: {small}/{len(rel_sizes)} "
              f"({100*small/len(rel_sizes):.1f}%) -> these count as 'small objects' in COCO terms")
        print("\n  Guidance: if a large share of boxes are <32px at native scale, avoid "
              "resizing those source images down to 640 before training (it shrinks them "
              "further) — tile large images first (see tile_large_images.py) or train at "
              "a larger --imgsz (960/1280).")


if __name__ == "__main__":
    main()


Overwriting analyze_dataset.py


## 4. Convert each dataset to YOLO format

In [24]:
import subprocess

def run(cmd):
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True)

run(["python", "convert_deeppcb.py", "--src", DEEPPCB_ROOT, "--out", f"{WORK}/converted/deeppcb"])

if HRIPCB_ANN and HRIPCB_IMG:
    run(["python", "convert_hripcb.py", "--ann", HRIPCB_ANN, "--img", HRIPCB_IMG, "--out", f"{WORK}/converted/hripcb"])
else:
    print("Skipping HRIPCB conversion — annotations or images not found (see Section 1 warnings).")

if PCBDEFECT_JSON and PCBDEFECT_IMG:
    run(["python", "convert_pcbdefect.py", "--json", PCBDEFECT_JSON, "--img", PCBDEFECT_IMG, "--out", f"{WORK}/converted/pcbdefect"])
else:
    print("Skipping PCB-Defect conversion — json or images not found (see Section 1 warnings).")


+ python convert_deeppcb.py --src /kaggle/input/datasets/rahatrihan/micro-pcb-defect/DeepPCB-master --out /kaggle/working/converted/deeppcb
[DeepPCB] converted 1500 images, 8512 boxes, skipped 3 (no annotation found)
+ python convert_hripcb.py --ann /kaggle/input/datasets/rahatrihan/micro-pcb-defect/Tiny-Defect-Detection-for-PCB-master/Tiny-Defect-Detection-for-PCB-master/tools/test_annotation --img /kaggle/input/datasets/rahatrihan/micro-pcb-defect/Tiny-Defect-Detection-for-PCB-master/Tiny-Defect-Detection-for-PCB-master/tools/inference_results --out /kaggle/working/converted/hripcb
[HRIPCB] converted 1 images, 3 boxes, skipped 1 (image not found)
+ python convert_pcbdefect.py --json /kaggle/input/datasets/rahatrihan/micro-pcb-defect/PCB_Defect/PCB_Defect/annotation/_annotations.coco.json --img /kaggle/input/datasets/rahatrihan/micro-pcb-defect/PCB_Defect/PCB_Defect/images --out /kaggle/working/converted/pcbdefect
[PCB-Defect] WARNING: unmapped category 'detecting-pcb-defects' — check

## 5. Tile the PCB-Defect images

PCB-Defect images run up to 6000x4000px - resizing straight to YOLO's 640 input
would shrink small defects to a few pixels. Tile into overlapping 1024px patches
instead. DeepPCB (640x640) and HRIPCB (~600x600) don't need this.

In [25]:
import os

if os.path.isdir(f"{WORK}/converted/pcbdefect"):
    run(["python", "tile_large_images.py",
         "--src", f"{WORK}/converted/pcbdefect",
         "--out", f"{WORK}/converted/pcbdefect_tiled",
         "--tile", "1024", "--overlap", "128", "--min-size", "1500"])
else:
    print("Skipping tiling — no converted PCB-Defect data.")


+ python tile_large_images.py --src /kaggle/working/converted/pcbdefect --out /kaggle/working/converted/pcbdefect_tiled --tile 1024 --overlap 128 --min-size 1500
[Tile] processed 230 source images -> 360 output images/tiles


## 6. Merge + split into train/val

Only datasets that were actually converted are included automatically.

In [26]:
sources = []
if os.path.isdir(f"{WORK}/converted/deeppcb"):
    sources.append(f"deeppcb={WORK}/converted/deeppcb")
if os.path.isdir(f"{WORK}/converted/hripcb"):
    sources.append(f"hripcb={WORK}/converted/hripcb")
if os.path.isdir(f"{WORK}/converted/pcbdefect_tiled"):
    sources.append(f"pcbdefect={WORK}/converted/pcbdefect_tiled")
elif os.path.isdir(f"{WORK}/converted/pcbdefect"):
    sources.append(f"pcbdefect={WORK}/converted/pcbdefect")

print("Merging sources:", sources)
run(["python", "merge_split.py", "--src", *sources,
     "--out", f"{WORK}/dataset", "--val-frac", "0.15", "--seed", "42"])


Merging sources: ['deeppcb=/kaggle/working/converted/deeppcb', 'hripcb=/kaggle/working/converted/hripcb', 'pcbdefect=/kaggle/working/converted/pcbdefect_tiled']
+ python merge_split.py --src deeppcb=/kaggle/working/converted/deeppcb hripcb=/kaggle/working/converted/hripcb pcbdefect=/kaggle/working/converted/pcbdefect_tiled --out /kaggle/working/dataset --val-frac 0.15 --seed 42
[deeppcb] train=1275  val=225
[hripcb] train=0  val=1
[pcbdefect] train=306  val=54

TOTAL: train=1581  val=280
Dataset root written to: /kaggle/working/dataset


## 7. Sanity-check before training (class balance + small-object share)

In [27]:
run(["python", "analyze_dataset.py", "--root", f"{WORK}/dataset", "--split", "train"])


+ python analyze_dataset.py --root /kaggle/working/dataset --split train
=== Images per source dataset ===
  deeppcb      1275
  pcbdefect    306

=== Box count per class ===
  missing_hole         1237  ( 16.4%)
  mouse_bite           1275  ( 16.9%)
  open_circuit            0  (  0.0%)
  short                1664  ( 22.0%)
  spur                 1988  ( 26.3%)
  spurious_copper      1396  ( 18.5%)

=== Box size (absolute pixels, native image scale) ===
  avg box size: 41.6 x 37.9 px
  boxes smaller than 32x32px: 838/7560 (11.1%) -> these count as 'small objects' in COCO terms

  Guidance: if a large share of boxes are <32px at native scale, avoid resizing those source images down to 640 before training (it shrinks them further) — tile large images first (see tile_large_images.py) or train at a larger --imgsz (960/1280).


## 8. Write data.yaml and train

Kaggle sessions have a GPU time quota and a 9-12h session limit, and
`/kaggle/working` output is capped (commonly ~20GB) - keep `epochs` reasonable
and let early stopping (`patience`) cut a run short once it plateaus.

In [28]:
data_yaml = f"""path: {WORK}/dataset
train: images/train
val: images/val

nc: 6
names:
  0: missing_hole
  1: mouse_bite
  2: open_circuit
  3: short
  4: spur
  5: spurious_copper
"""
with open(f"{WORK}/data.yaml", "w") as f:
    f.write(data_yaml)
print(data_yaml)


path: /kaggle/working/dataset
train: images/train
val: images/val

nc: 6
names:
  0: missing_hole
  1: mouse_bite
  2: open_circuit
  3: short
  4: spur
  5: spurious_copper



**`imgsz` guidance** (see step 7's output for your actual small-object share):

| Situation | imgsz |
|---|---|
| Baseline / limited GPU memory | 640 |
| Recommended default for this merged set | **960** |
| Best small-defect recall (if GPU allows) | 1280 (reduce `batch` accordingly) |


In [29]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")  # swap for yolov8n.pt if you need faster/lighter, or yolov8m.pt if GPU allows
results = model.train(
    data=f"{WORK}/data.yaml",
    imgsz=960,
    epochs=150,
    batch=16,
    patience=30,
    project=f"{WORK}/runs",
    name="pcb_yolo",
)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.4.144 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.143 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=Fa

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/150      7.36G      1.384      1.265       1.12         88        960: 100% ━━━━━━━━━━━━ 99/99 2.4it/s 41.3s0.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 2.4it/s 3.7s0.4ss
                   all        280       1331      0.834      0.739      0.818      0.354

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/150      7.32G      1.358       1.09      1.122        100        960: 100% ━━━━━━━━━━━━ 99/99 2.4it/s 42.0s0.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 2.5it/s 3.6s0.4ss
                   all        280       1331       0.81      0.776      0.794      0.264

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/150      7.38G      1.297     0.9937      1.105         84        960: 100% ━━━━━━━━━━━━ 99/99 2.4it/s 41.6s0.4ss
                 Class     Images  Ins

!find /kaggle/working -name "*.pt" 2>/dev/null
!ls -la /kaggle/working/runs/pcb_yolo/weights/ 2>/dev/null

In [4]:
!find /kaggle/working -name "*.pt" 2>/dev/null
!ls -la /kaggle/working/runs/pcb_yolo/weights/ 2>/dev/null

In [5]:
import shutil
shutil.copy("/kaggle/working/runs/pcb_yolo/weights/best.pt", "/kaggle/working/pcb_yolo_best.pt")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/runs/pcb_yolo/weights/best.pt'

## 9. Validate

In [2]:
!pip install -q ultralytics
from ultralytics import YOLO

model = YOLO("/kaggle/working/runs/pcb_yolo/weights/best.pt")
print(model.names)  # confirms your 6 classes are loaded correctly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/runs/pcb_yolo/weights/best.pt'

In [30]:
metrics = model.val(data=f"{WORK}/data.yaml")
print(metrics.box.map, metrics.box.map50)


Ultralytics 8.4.143 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 964.6±365.6 MB/s, size: 45.7 KB)
val: Scanning /kaggle/working/dataset/labels/val.cache... 280 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 280/280 106.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.1it/s 8.4s0.4s
                   all        280       1331      0.977      0.967      0.981      0.784
          missing_hole        203        240          1      0.962      0.975      0.924
            mouse_bite        170        231      0.943      0.948      0.982      0.717
                 short        200        278      0.973          1      0.988      0.742
                  spur        249        353       0.98      0.949      0.969       0.76
       spurious_copper        164    

## Notes

- **HRIPCB data gap**: if Section 1 warned about 0 XML files, your submitted
  `Tiny-Defect-Detection-for-PCB-master.zip` is code only — you're training on
  DeepPCB + PCB-Defect until you source the actual HRIPCB images/annotations
  and add them as a separate input.
- **Session limits**: GPU sessions are capped (commonly ~30h/week quota,
  ~9-12h per session on the free tier) - if training is cut off, resume from
  `/kaggle/working/runs/pcb_yolo/weights/last.pt` with `model = YOLO(".../last.pt"); model.train(resume=True)`.
- **Output size cap**: delete `converted/` after step 6 if space is tight:
  `!rm -rf {WORK}/converted`.
- **class_map.py**: double-check `DEEPPCB_ID_TO_CANONICAL` against the README
  bundled with your DeepPCB-master zip — the numeric defect-id ordering has
  been reported inconsistently across mirrors.
